# 12 – Ensemble of E3 (Elia) and V4 (Fatemeh)

**Question:** does averaging the predictions of our two best models beat the best single one?

| model | augmentation | OOF clean | OOF shifted | Kaggle public |
|---|---|---|---|---|
| **E3** `e3_hue_blur` (notebook 11) | dihedral + hue rotation + mild blur | 0.847 | 0.846 | 0.8574 |
| **V4** Fatemeh, shared split | free rotation, zoom, shift, strong colour, strong blur, lines/patches, class weights, 25 epochs | 0.859 | 0.851 | 0.8678 |

Both models use the **same 2,995 images and the same 5 folds**. Every image therefore has an out-of-fold (OOF) prediction from both models, made by models that never saw it, and the ensemble can be evaluated **without training anything**: we only average probabilities that are already saved.

**Why it can help.** An ensemble gains when the models make **different** mistakes: where one is wrong and the other is right and confident, the average can fix the prediction. E3 and V4 are trained with very different augmentations, so their mistakes should differ more than E3's and E4's.

**How:** for each image, `p = w · p_V4 + (1 − w) · p_E3` (probabilities of the 10 classes), then the predicted class is the one with the highest `p`.
- The main candidate is **w = 0.5** (plain average).
- We also look at the whole curve w = 0 … 1, but choosing w on the OOF slightly overfits it. We move away from 0.5 only if another weight is clearly better (≳ 0.005).

**Nothing is trained here:** the notebook runs on the CPU in seconds.

**Inputs needed** (the V4 files come from the *Output* tab of Fatemeh's Kaggle notebook version, the one that scored 0.8678). Put these 5 files in `families/elia/outputs/v4_fatemeh/`:
- `oof_clean.npy`
- `oof_shifted.npy`
- `oof_ids.csv`
- `test_probs.npy`
- `submission.csv`

The E3 files are already in `outputs/e3_hue_blur/` (from notebook 11).

## 0. Setup and loading

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

import train_lib as T
import common as C

plt.rcParams["figure.dpi"] = 80
E3, V4, ENS = "e3_hue_blur", "v4_fatemeh", "ensemble_e3_v4"
OUT = T.OUT_DIR
K = C.N_CLASSES
P_COLS = lambda name: [f"p{c}_{name}" for c in range(K)]

# --- E3 (notebook 11): oof.csv with id, label, fold, p0_clean.., p0_shifted..; test_probs (folds, N, 10) ---
e3 = pd.read_csv(OUT / E3 / "oof.csv", dtype={"id": str})
e3_test = np.load(OUT / E3 / "test_probs.npy")
e3_test = e3_test.mean(0) if e3_test.ndim == 3 else e3_test          # mean of the 5 fold models

# --- V4 (Fatemeh, Kaggle): oof_clean / oof_shifted (2995, 10) in the row order of oof_ids.csv; test_probs (N, 10) ---
v4_dir = OUT / V4
v4_ids = pd.read_csv(v4_dir / "oof_ids.csv", dtype={"id": str})
v4_clean, v4_shift = np.load(v4_dir / "oof_clean.npy"), np.load(v4_dir / "oof_shifted.npy")
v4_test = np.load(v4_dir / "test_probs.npy")
v4_sub = pd.read_csv(v4_dir / "submission.csv", dtype={"id": str})

test_ids = pd.read_csv(C.DATA / "sample_submission.csv", dtype={"id": str}).id.tolist()
print("E3 OOF", e3.shape, "| E3 test", e3_test.shape)
print("V4 OOF", v4_clean.shape, v4_shift.shape, "| V4 test", v4_test.shape)

## 1. Sanity checks

Before averaging, we check that the two models talk about the **same images in the same order**: same ids, same labels and same folds in the OOF, and the same test order. We also recompute each model's scores to verify that we are reading the right files:
- **E3:** 0.8467 clean / 0.8458 shifted;
- **V4:** 0.8587 clean / 0.8513 shifted.

The V4 rows are re-ordered to E3's order through the ids, so the two do not need to be saved in the same order.

In [ ]:
# OOF: align V4 to E3's row order through the ids
pos = pd.Series(np.arange(len(v4_ids)), index=v4_ids.id)
assert set(v4_ids.id) == set(e3.id) and len(v4_ids) == len(e3) == 2995, "the two OOF files do not cover the same images"
order = pos.loc[e3.id].to_numpy()
v4_ids, v4_clean, v4_shift = v4_ids.iloc[order].reset_index(drop=True), v4_clean[order], v4_shift[order]
assert (v4_ids.label.to_numpy() == e3.label.to_numpy()).all(), "labels differ"
assert (v4_ids.fold.to_numpy() == e3.fold.to_numpy()).all(), "folds differ: the models were not trained on the same split"
assert not np.isnan(v4_clean).any() and not np.isnan(v4_shift).any(), "V4 OOF is incomplete (QUICK run?)"

# test: same order as sample_submission
assert v4_test.shape == e3_test.shape == (len(test_ids), K)
assert v4_sub.id.tolist() == test_ids, "V4 test order differs from sample_submission"
assert (v4_sub.label.to_numpy() == v4_test.argmax(1)).all(), "V4 test_probs.npy does not match its submission.csv"

y, folds = e3.label.to_numpy(), e3.fold.to_numpy()
PROBS = {   # name -> (clean OOF, shifted OOF, test)
    "E3": (e3[P_COLS("clean")].to_numpy(), e3[P_COLS("shifted")].to_numpy(), e3_test),
    "V4": (v4_clean, v4_shift, v4_test),
}
f1w = lambda P: f1_score(y, P.argmax(1), average="weighted")
for name, (pc, ps, _) in PROBS.items():
    print(f"{name}: OOF F1 clean {f1w(pc):.4f} | shifted {f1w(ps):.4f}")

## 2. Do the two models make different mistakes?

An ensemble can only fix an image if **at least one** model gets it right. The table splits the OOF images into four groups:
- both right;
- only E3 right;
- only V4 right;
- both wrong.

"Both wrong" is roughly the floor of the error: averaging cannot fix those images, except in rare cases where the correct class wins only on the average. The two "only one right" groups are where the ensemble can gain, if the right model is more confident than the wrong one.

In [ ]:
rows = {}
for val, i in [("clean", 0), ("shifted", 1)]:
    r3 = PROBS["E3"][i].argmax(1) == y
    r4 = PROBS["V4"][i].argmax(1) == y
    rows[val] = {"both right": (r3 & r4).mean(), "only E3 right": (r3 & ~r4).mean(),
                 "only V4 right": (~r3 & r4).mean(), "both wrong": (~r3 & ~r4).mean(),
                 "same prediction": (PROBS["E3"][i].argmax(1) == PROBS["V4"][i].argmax(1)).mean()}
(pd.DataFrame(rows).T * 100).round(1).astype(str) + " %"

## 3. Ensemble weight: from E3 alone (w = 0) to V4 alone (w = 1)

For each weight `w` we average the probabilities and compute the OOF weighted F1 on both validations.

In [ ]:
W = np.round(np.linspace(0, 1, 11), 2)
blend = lambda w, i: w * PROBS["V4"][i] + (1 - w) * PROBS["E3"][i]
curve = pd.DataFrame({"w (V4 weight)": W,
                      "F1 clean": [f1w(blend(w, 0)) for w in W],
                      "F1 shifted": [f1w(blend(w, 1)) for w in W]}).set_index("w (V4 weight)")

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(curve.index, curve["F1 clean"], "o-", label="clean")
ax.plot(curve.index, curve["F1 shifted"], "o-", label="shifted")
ax.axvline(0.5, color="gray", ls=":", lw=1)
ax.set_xlabel("weight of V4  (0 = E3 alone, 1 = V4 alone)"); ax.set_ylabel("OOF weighted F1"); ax.legend()
ax.set_title("Ensemble E3 + V4: OOF F1 vs weight")
plt.tight_layout(); plt.show()
curve.style.format("{:.4f}").highlight_max(color="#c6efce")

## 4. Per fold: is the gain consistent?

As in notebook 11, we compare on the **same fold**:
- the plain average (w = 0.5);
- the best single model (V4).

The ensemble is convincing if it wins on most folds, not only on average.

In [ ]:
W_ENS = 0.5                       # change only if section 3 shows a clearly better weight (≳ 0.005)
rows = []
for k in range(5):
    m = folds == k
    f = lambda P: f1_score(y[m], P[m].argmax(1), average="weighted")
    rows.append({"fold": k,
                 "E3 shifted": f(PROBS["E3"][1]), "V4 shifted": f(PROBS["V4"][1]), "ENS shifted": f(blend(W_ENS, 1)),
                 "E3 clean": f(PROBS["E3"][0]), "V4 clean": f(PROBS["V4"][0]), "ENS clean": f(blend(W_ENS, 0))})
pf = pd.DataFrame(rows).set_index("fold")
pf["ENS − V4 shifted"] = pf["ENS shifted"] - pf["V4 shifted"]
pf["ENS − V4 clean"] = pf["ENS clean"] - pf["V4 clean"]
print(f"folds where the ensemble beats V4: shifted {(pf['ENS − V4 shifted'] > 0).sum()}/5 | "
      f"clean {(pf['ENS − V4 clean'] > 0).sum()}/5")
pf.style.format("{:.4f}").background_gradient(subset=["ENS − V4 shifted", "ENS − V4 clean"], cmap="RdYlGn", vmin=-0.02, vmax=0.02)

## 5. Save the ensemble and compare with the other runs

We save the ensemble in the **same format** as the experiments of notebook 11, in `outputs/ensemble_e3_v4/`:
- `oof.csv`, the OOF probabilities;
- `test_probs.npy`;
- `submission.csv`.

V4 is saved in the same format too (`outputs/v4_fatemeh/oof.csv`), so `T.compare` and `T.per_class` work for all of them.

In [ ]:
def save_oof(name, pc, ps):
    df = e3[["id", "label", "fold"]].copy()
    for c in range(K):
        df[f"p{c}_clean"] = pc[:, c]
    for c in range(K):
        df[f"p{c}_shifted"] = ps[:, c]
    (OUT / name).mkdir(parents=True, exist_ok=True)
    df.to_csv(OUT / name / "oof.csv", index=False)

save_oof(V4, v4_clean, v4_shift)
ens_clean, ens_shift = blend(W_ENS, 0), blend(W_ENS, 1)
ens_test = W_ENS * v4_test + (1 - W_ENS) * e3_test
save_oof(ENS, ens_clean, ens_shift)
np.save(OUT / ENS / "test_probs.npy", ens_test)
sub = pd.DataFrame({"id": test_ids, "label": ens_test.argmax(1)})
sub.to_csv(OUT / ENS / "submission.csv", index=False)
assert len(sub) == 14736 and sub.label.between(0, 9).all()
print("saved", OUT / ENS / "submission.csv")

table = T.compare([E3, V4, ENS])
table.style.format("{:.4f}").background_gradient(subset=["Δ shifted"], cmap="RdYlGn", vmin=-0.04, vmax=0.04)

In [ ]:
names = [E3, V4, ENS]
print("SHIFTED validation"); display(T.per_class(names, "shifted").round(3))
print("CLEAN validation");   display(T.per_class(names, "clean").round(3))

## 6. On the test set: how many predictions change?

On the test we have no labels. We can only count how often the ensemble disagrees with each model, and see which classes the changes go to. If the ensemble changes very few predictions compared with V4, its Kaggle score cannot differ much from V4's.

In [ ]:
p3, p4, pe = e3_test.argmax(1), v4_test.argmax(1), ens_test.argmax(1)
print(f"test images: {len(pe)}")
print(f"E3 vs V4 disagree on      {(p3 != p4).mean():6.1%}")
print(f"ensemble differs from V4 on {(pe != p4).mean():6.1%}  ({(pe != p4).sum()} images)")
print(f"ensemble differs from E3 on {(pe != p3).mean():6.1%}")
mix = pd.DataFrame({"train %": np.bincount(y, minlength=K) / len(y) * 100,
                    "E3 %": np.bincount(p3, minlength=K) / len(p3) * 100,
                    "V4 %": np.bincount(p4, minlength=K) / len(p4) * 100,
                    "ensemble %": np.bincount(pe, minlength=K) / len(pe) * 100},
                   index=[f"{c} {C.CLASSES[c]}" for c in range(K)]).round(1)
mix.T

## 7. Decision

Rule (same as phase 3, with V4 as the reference to beat):
- the ensemble is worth submitting if its OOF beats **V4** by ≳ 0.005 on the shifted validation and wins on most folds;
- otherwise V4 alone stays the best, and the ensemble does not justify its double cost.

If it passes: submit `outputs/ensemble_e3_v4/submission.csv` and write the score in the log.

Results (fill in):
- OOF ensemble (w = ___): clean ___ | shifted ___ (V4: 0.8587 | 0.8513)
- folds won vs V4: shifted ___/5, clean ___/5
- Kaggle public: ___ (V4: 0.8678)